### Mask coherence with shadow and layover and incidence angle 

In [3]:
import rasterio
from rasterio.merge import merge
import numpy as np
from pathlib import Path
import glob
from rasterio.warp import reproject, Resampling

from collections import defaultdict

In [4]:
# load shadow and layover mask
BASE = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/S1_LIA/"

In [5]:
shadow_1 = BASE + "S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F/S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F/S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F_ls_map.tif"
shadow_2 = BASE + "S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8/S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8/S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8_ls_map.tif"

# load both and create mosaic
with rasterio.open(shadow_1) as src1, rasterio.open(shadow_2) as src2:
    mosaic, shadow_transform = merge([src1, src2], method="max")
    shadow_mosaic = mosaic[0]
    shadow_profile = src1.profile
    shadow_crs = src1.crs

# save the mosaic to a new file
output_shadow_mosaic = BASE + "shadow_mosaic.tif"
shadow_profile.update({
    "height": shadow_mosaic.shape[0],
    "width": shadow_mosaic.shape[1],
    "transform": shadow_transform,
    "crs": shadow_crs
})

In [6]:
# load incidence angle image
incidence_1 = BASE + "S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F/S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F/S1A_IW_20230807T052734_DVP_RTC10_G_gpuned_6A3F_inc_map.tif"
incidence_2 = BASE + "S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8/S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8/S1A_IW_20230807T052709_DVP_RTC10_G_gpuned_8CD8_inc_map.tif"

# load both and create mosaic
with rasterio.open(incidence_1) as src1, rasterio.open(incidence_2) as src2:
    mosaic, incidence_transform = merge([src1, src2], method="max")
    incidence_mosaic = mosaic[0]
    incidence_profile = src1.profile
    incidence_crs = src1.crs

# save the mosaic to a new file
output_incidence_mosaic = BASE + "incidence_mosaic.tif"
incidence_profile.update({
    "height": incidence_mosaic.shape[0],
    "width": incidence_mosaic.shape[1],
    "transform": incidence_transform,
    "crs": incidence_crs
})

### Load Sentinel-1 coherence images and mask them

In [9]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.merge import merge
from rasterio.io import MemoryFile

INPUT_FOLDER = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/")
YEARS = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"]

def align(mosaic, m_transform, m_crs, profile, shape):
    out = np.zeros(shape, dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=out,
        src_transform=m_transform,
        src_crs=m_crs,
        dst_transform=profile["transform"],
        dst_crs=profile["crs"],
        resampling=Resampling.nearest,
    )
    return out

for year in YEARS:
    year_folder = INPUT_FOLDER / year
    composites = sorted(year_folder.glob(f"sar_coherence_T*_{year}_composite.tiff"))
    print(f"{year_folder}: {len(composites)} composites")

    for f in composites:
        print(f"Masking {f.name}")

        with rasterio.open(f) as src:
            img = src.read(1)
            profile = src.profile.copy()

        # 1. Align incidence angle and shadow/layover mask to the composite grid
        #aligned_incidence = align(incidence_mosaic, incidence_transform, incidence_crs, profile, img.shape)
        aligned_shadow    = align(shadow_mosaic,    shadow_transform,    shadow_crs,    profile, img.shape)

        # 2. Valid pixels
        valid = aligned_shadow == 1
        #(aligned_incidence > 0.1) & (aligned_incidence < 0.8) 

        # 3. Apply mask
        masked = img.copy()
        masked[~valid] = 0

        # 4. Save (new file, so the original composite is preserved)
        profile.update(nodata=0)
        output_file = f.with_name(f.stem + "_masked_shadow.tiff")
        print(f"Saving to {output_file}")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(masked, 1)

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2016: 5 composites
Masking sar_coherence_T117_2016_composite.tiff


/tmp/ipykernel_6750/2908575000.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  img = src.read(1)


Saving to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2016/sar_coherence_T117_2016_composite_masked_shadow.tiff
Masking sar_coherence_T15_2016_composite.tiff


/tmp/ipykernel_6750/2908575000.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  img = src.read(1)


Saving to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2016/sar_coherence_T15_2016_composite_masked_shadow.tiff
Masking sar_coherence_T168_2016_composite.tiff


/tmp/ipykernel_6750/2908575000.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  img = src.read(1)


... (101 outputs left out)

/tmp/ipykernel_6750/2908575000.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  img = src.read(1)


Saving to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2026/sar_coherence_T44_2026_composite_masked_shadow.tiff
Masking sar_coherence_T95_2026_composite.tiff


/tmp/ipykernel_6750/2908575000.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  img = src.read(1)


Saving to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2026/sar_coherence_T95_2026_composite_masked_shadow.tiff


In [10]:
import re
from pathlib import Path
import rasterio
from rasterio.merge import merge

YEARS = list(range(2016, 2027))
BASE_OUTPUT_FOLDER = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence")

ASCENDING_ORBITS = {15, 44, 117}
DESCENDING_ORBITS = {95, 168}


def merge_swaths(files, output_filename):
    datasets = [rasterio.open(f) for f in files]
    try:
        mosaic, transform = merge(datasets, method="max", nodata=0)
        meta = datasets[0].meta.copy()
    finally:
        for ds in datasets:
            ds.close()

    meta.update(
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        count=1,
        nodata=0,
        dtype=mosaic.dtype,
    )

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(mosaic[0], 1)

    print(f"Saved: {output_filename}")


for year in YEARS:
    year_folder = BASE_OUTPUT_FOLDER / str(year)

    if not year_folder.exists():
        print(f"{year}: folder not found, skipping")
        continue

    files = sorted(year_folder.glob(f"sar_coherence_T*_{year}_composite_masked_shadow.tiff"))

    ascending_files, descending_files = [], []
    for f in files:
        match = re.search(r"T(\d+)", f.name)
        if not match:
            continue
        orbit = int(match.group(1))
        if orbit in ASCENDING_ORBITS:
            ascending_files.append(f)
        elif orbit in DESCENDING_ORBITS:
            descending_files.append(f)

    print(f"{year}: {len(ascending_files)} ascending, {len(descending_files)} descending")

    if ascending_files:
        merge_swaths(ascending_files, year_folder / f"coh_mosaic_ascending_{year}_masked_shadow.tiff")
    else:
        print(f"{year}: no ascending files, skipping")

    if descending_files:
        merge_swaths(descending_files, year_folder / f"coh_mosaic_descending_{year}_masked_shadow.tiff")
    else:
        print(f"{year}: no descending files, skipping")

2016: 3 ascending, 2 descending
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2016/coh_mosaic_ascending_2016_masked_shadow.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2016/coh_mosaic_descending_2016_masked_shadow.tiff
2017: 3 ascending, 2 descending
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2017/coh_mosaic_ascending_2017_masked_shadow.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2017/coh_mosaic_descending_2017_masked_shadow.tiff
2018: 3 ascending, 2 descending
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2018/coh_mosaic_ascending_2018_masked_shadow.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence/2018/coh_mosaic_descending_2018_masked_shadow.tiff
2019: 3 ascending, 2 descending
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/Gla

In [ ]:
# plot one result of the masked mosaic
